## Statistical tests

--> Sharpe (over time), Annual returns (over time)
1. Compare with existing methods performance (Equal weight, MVO, DFL, LLM)
2. Generate a number of random portfolios (then test for statistical significance above noise, i.e. null hypothesis)
3. Use a perturbation test (for robustness)
4. Stress data (regime fitting)

In [ ]:
import math
import numpy as np

from algorithm.data_caller import get_data, normalize_wrds_data
from backtester.get_performance import track_random_performance, visualize_performance

raw_data = get_data()
normalized_data = normalize_wrds_data(raw_data)

random_returns, random_annualized, random_sharpe = track_random_performance(
    raw_data,
    asset_cap=30,
    iterations=50,
    timesteps=100,
    seed=42,
)

visualize_performance(random_returns, random_sharpe, random_annualized)


def compare_random(metric):
    metric = np.asarray(metric, dtype=float)
    mean = float(np.mean(metric))
    std = float(np.std(metric, ddof=0))

    if std == 0.0:
        return 1.0, (mean, mean)

    z_scores = (metric - mean) / std
    p_value = 1.0 - 0.5 * (1.0 + math.erf(np.max(np.abs(z_scores)) / math.sqrt(2.0)))
    c_interval = (mean - 1.96 * std, mean + 1.96 * std)
    return p_value, c_interval